# AI 애플리케이션 배포

## 준비 사항
- 벡터 저장소 : 슈파 베이스
- 모니터링 및 디버깅 : 랭스미스
- 백엔드 API : 랭그래프 플랫폼

### 종속성 설치
`.env` 추가 및 라이브러리 설치

### LLM
api 발급
- https://platform.openai.com/settings/organization/api-keys


### 벡터 저장소
- supabase
  - project : https://supabase.com/dashboard/project/isgsnhxthnlymkrigtng  
  - sql : https://supabase.com/dashboard/porject/isgsnhxthnlymkrigtng/sql/3b819ca4-f2fc-467f-b8a2-1cc36d33968e

#### 슈파베이스 임베딩 테스트


In [5]:
import { SupabaseVectorStore } from 'npm:@langchain/community/vectorstores/supabase';
import { OpenAIEmbeddings } from 'npm:@langchain/openai';

import {Document}from 'npm:@langchain/core/documents';
import {createClient} from 'npm:@supabase/supabase-js';
import dotenv from 'npm:dotenv';


dotenv.config();

function omit(s, len=20) {
  if (!s) return s
  return s.slice(0, len) + '...'
}

console.log(`OPEN_API_KEY=${omit(process.env.OEPN_API_KEY)}`);

const embeddings = new OpenAIEmbeddings({
  openAIApiKey: process.env.OEPN_API_KEY,
});


const supabaseUrl = process.env.SUPABASE_URL;
const supabaseKey = process.env.SUPABASE_SERVICE_ROLE_KEY;

const supabaseClient = createClient(supabaseUrl, supabaseKey);

const vectorStore = new SupabaseVectorStore(embeddings, {
  client: supabaseClient,
  tableName: 'documents',
  queryName: 'match_documents',
});

const document1: Document = new Document({
  pageContent: 'The powerhouse of the cell is the mitochondria',
  metadata: { source: 'https://example.com' },
});

const document2: Document = new Document({
  pageContent: 'Buildings are made of bricks',
  metadata: { source: 'https://example.com' },
});

const documents = [document1, document2];

// 데이터베이스에 데이터 저장
await vectorStore.addDocuments(documents, {
  ids: ['1', '2'],
});

// 벡터 저장소에 쿼리 전송

const filter = {
  source: 'https://example.com',
};

const similaritySearchResults = await vectorStore.similaritySearch("biology", 2, filter);

for (const result of similaritySearchResults) {
  console.log(`* ${result.pageContent} [${JSON.stringify(result.metadata, null)}]`);
}



[dotenv@17.2.2] injecting env (6) from .env -- tip: ⚙️  enable debug logging with { debug: true }
OPEN_API_KEY=sk-proj-NVmaKRJNkZZz...
* The powerhouse of the cell is the mitochondria [{"source":"https://example.com"}]
* Buildings are made of bricks [{"source":"https://example.com"}]


- 겪은 이슈
  - open api key 설정 제대로 안됨 --> 로컬 실행 했을때..
    - option 추가
```typescript
const embeddings = new OpenAIEmbeddings({
  openAIApiKey: process.env.OEPN_API_KEY,
});
```

```
file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+openai@0.3.17_@langchain+core@0.3.77_openai@4.104.0_ws@8.18.3_zod@3.25.76___ws@8.18.3/node_modules/@langchain/openai/dist/embeddings.js:128
            throw new Error("OpenAI or Azure OpenAI API key or Token Provider not found");
                  ^

Error: OpenAI or Azure OpenAI API key or Token Provider not found
    at new OpenAIEmbeddings (file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+openai@0.3.17_@langchain+core@0.3.77_openai@4.104.0_ws@8.18.3_zod@3.25.76___ws@8.18.3/node_modules/@langchain/openai/dist/embeddings.js:128:19)
    at file:///Users/user/hs-dev/learning-langchain/ch9/js/src/supabasStore.ts:10:20
    at ModuleJob.run (node:internal/modules/esm/module_job:371:25)
    at async onImport.tracePromise.__proto__ (node:internal/modules/esm/loader:702:26)
    at async asyncRunEntryPointWithESMLoader (node:internal/modules/run_main:101:5)
```

  - open api key billing 오류...
    - 코인 구매..(?)
```
file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+openai@0.3.17_@langchain+core@0.3.77_openai@4.104.0_ws@8.18.3_zod@3.25.76___ws@8.18.3/node_modules/@langchain/openai/dist/embeddings.js:128
            throw new Error("OpenAI or Azure OpenAI API key or Token Provider not found");
                  ^

Error: OpenAI or Azure OpenAI API key or Token Provider not found
    at new OpenAIEmbeddings (file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+openai@0.3.17_@langchain+core@0.3.77_openai@4.104.0_ws@8.18.3_zod@3.25.76___ws@8.18.3/node_modules/@langchain/openai/dist/embeddings.js:128:19)
```

  - sql error
    - cursor에 물어봐서 오타 찾음...
```
file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+community@0.3.56_@browserbasehq+sdk@2.6.0_@browserbasehq+stagehand@1.14.0_@p_b8f17fe88e66818ff28505c6b02041a5/node_modules/@langchain/community/dist/vectorstores/supabase.js:256
            throw new Error(`Error searching for documents: ${error.code} ${error.message} ${error.details}`);
                  ^

Error: Error searching for documents: PGRST202 Could not find the function public.match_documents(filter, match_count, query_embedding) in the schema cache Searched for the function public.match_documents with parameters filter, match_count, query_embedding or with a single unnamed json/jsonb parameter, but no matches were found in the schema cache.
    at SupabaseVectorStore._searchSupabase (file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+community@0.3.56_@browserbasehq+sdk@2.6.0_@browserbasehq+stagehand@1.14.0_@p_b8f17fe88e66818ff28505c6b02041a5/node_modules/@langchain/community/dist/vectorstores/supabase.js:256:19)
    at process.processTicksAndRejections (node:internal/process/task_queues:105:5)
    at async SupabaseVectorStore.similaritySearchVectorWithScore (file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+community@0.3.56_@browserbasehq+sdk@2.6.0_@browserbasehq+stagehand@1.14.0_@p_b8f17fe88e66818ff28505c6b02041a5/node_modules/@langchain/community/dist/vectorstores/supabase.js:268:26)
    at async SupabaseVectorStore.similaritySearch (file:///Users/user/hs-dev/learning-langchain/ch9/js/node_modules/.pnpm/@langchain+core@0.3.77_openai@4.104.0_ws@8.18.3_zod@3.25.76_/node_modules/@langchain/core/dist/vectorstores.js:260:25)
    at async file:///Users/user/hs-dev/learning-langchain/ch9/js/src/supabasStore.ts:61:33
```



### 벡엔드 API
- 다수의 동시 사용자 요구에 대응하고 대용량 상태 및 스레드를 효율적으로 저장할 수 있또록 수평 확장이 가능한 작업 큐와 서버, 강력한 Postgres 체크포인터를 관리
- 랭그래프 플랫폼 주요기능
  - 스트리밍 기능 및 사용자 개입 
  - 그외
    - 진행 중인 그래프 스레드에서 새로우 ㄴ사용자 입력을 효과적으로 처리하는 이중 전송
    - 오래 걸리는 작업의 비동기 백그라운드 처리
    - 정해진 일저에 따라 기본적인 업무를 자동으로 수행하는 크론 작업

#### 랭스미스
통합 개발자 플랫폼으로 디버깅, 협업, 테스트 및 LLM 애플리케이션 모니터링

### 랭그래프 플랫폼 API 이해하기
#### 데이터 모델
- 어시스턴트 
  - CompiledGraph 기반 인스턴스
- 스레드
- 실행
- 크론 잡

#### 기능
- 스트리밍
- 사용자 개입
- 이중 텍스트 전송
- 무상태 실행
- 웹훅


### 배포
#### 로컬
- langgraph-cli 설치 `pnpm add @langchain/langgraph-cli`
- 실행 `npx @langchain/langgraph-cli dev`


```bash
~ ❯ curl -X POST \                                                                                                                           
--url http://localhost:2024/runs/stream \
--header 'Content-Type: application/json' \
--data '{
  "assistant_id": "ingestion_graph",
  "input": {
    "messages": [{
      "role": "user",
      "content": "미국의 제30대 대통령이 사망했을 때 몇 살이었나요?"
    }]
  },
  "metadata": {},
  "config": {
    "configurable": {
        "useSampleDocs": true
    }
  },
  "multitask_strategy": "reject",
  "stream_mode": ["values"]
}'
event: metadata
data: {
data:   "run_id": "196bcdb3-822e-422b-a693-44fe039d6948",
data:   "attempt": 1
data: }

event: values
data: {
data:   "docs": []
data: }
```

```bash
❯ curl -X POST \ 
--url http://localhost:2024/runs/stream \
--header 'Content-Type: application/json' \
--data '{
  "assistant_id": "retrieval_graph",
  "input": {
    "query": "What is this document about?"
  },
  "metadata": {},
  "config": {
    "configurable": {
    }
  },
  "multitask_strategy": "reject",
  "stream_mode": ["values"]
}'
event: metadata
data: {
data:   "run_id": "8efed46a-d960-43d3-846a-0968da34430e",
data:   "attempt": 1
data: }

event: values
data: {
data:   "query": "What is this document about?",
data:   "messages": [],
data:   "documents": []
data: }

event: values
data: {
data:   "query": "What is this document about?",
data:   "route": "retrieve",
data:   "messages": [],
data:   "documents": []
data: }

event: values
data: {
data:   "query": "What is this document about?",
data:   "route": "retrieve",
data:   "messages": [],
data:   "documents": [
data:     {
data:       "pageContent": "Exhibit\nNumber\nIncorporated\tby\tReference\nFiled\nHerewithExhibit\tDescriptionFormFile\tNo.ExhibitFiling\tDate\n10.58††Grant\tContract\tfor\tState-Owned\tConstruction\tLand\nUse\tRight,\tdated\tas\tof\tOctober\t17,\t2018,\tby\tand\nbetween\tShanghai\tPlanning\tand\tLand\tResource\nAdministration\tBureau,\tas\tgrantor,\tand\tTesla\n(Shanghai)\tCo.,\tLtd.,\tas\tgrantee\t(English\ntranslation).\n10-Q001-3475610.2July\t29,\t2019\n10.59Credit\tAgreement,\tdated\tas\tof\tJanuary\t20,\t2023,\namong\tTesla,\tInc.,\tthe\tLenders\tand\tIssuing\tBanks\nfrom\ttime\tto\ttime\tparty\tthereto,\tCitibank,\tN.A.,\tas\nAdministrative\tAgent\tand\tDeutsche\tBank\nSecurities,\tInc.,\tas\tSyndication\tAgent\n10-K001-3475610.59January\t31,\t2023\n21.1List\tof\tSubsidiaries\tof\tthe\tRegistrant————X\n23.1Consent\tof\tPricewaterhouseCoopers\tLLP,\nIndependent\tRegistered\tPublic\tAccounting\tFirm\n————X\n31.1Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tExecutive\tOfficer\n————X\n31.2Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tFinancial\tOfficer\n————X\n32.1*Section\t1350\tCertifications————X\n97Tesla,\tInc.\tClawback\tPolicy————X\n101.INSInline\tXBRL\tInstance\tDocument————X\n101.SCHInline\tXBRL\tTaxonomy\tExtension\tSchema\nDocument\n————X\n101.CALInline\tXBRL\tTaxonomy\tExtension\tCalculation\nLinkbase\tDocument.\n————X\n101.DEFInline\tXBRL\tTaxonomy\tExtension\tDefinition\nLinkbase\tDocument\n————X\n101.LABInline\tXBRL\tTaxonomy\tExtension\tLabel\tLinkbase\nDocument\n————X\n101.PREInline\tXBRL\tTaxonomy\tExtension\tPresentation\nLinkbase\tDocument\n————X\n104Cover\tPage\tInteractive\tData\tFile\t(formatted\tas\ninline\tXBRL\twith\tapplicable\ttaxonomy\textension\ninformation\tcontained\tin\tExhibits\t101)\n*Furnished\therewith\n**Indicates\ta\tmanagement\tcontract\tor\tcompensatory\tplan\tor\tarrangement\n†Confidential\ttreatment\thas\tbeen\trequested\tfor\tportions\tof\tthis\texhibit\n††Portions\tof\tthis\texhibit\thave\tbeen\tredacted\tin\tcompliance\twith\tRegulation\tS-K\tItem\t601(b)(10).\n110",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 54,
data:             "from": 1
data:           },
data:           "pageNumber": 112
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "8717aa08-52f2-4e9f-a857-4fd69d28028c",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "Exhibit\nNumber\nIncorporated\tby\tReference\nFiled\nHerewithExhibit\tDescriptionFormFile\tNo.ExhibitFiling\tDate\n10.58††Grant\tContract\tfor\tState-Owned\tConstruction\tLand\nUse\tRight,\tdated\tas\tof\tOctober\t17,\t2018,\tby\tand\nbetween\tShanghai\tPlanning\tand\tLand\tResource\nAdministration\tBureau,\tas\tgrantor,\tand\tTesla\n(Shanghai)\tCo.,\tLtd.,\tas\tgrantee\t(English\ntranslation).\n10-Q001-3475610.2July\t29,\t2019\n10.59Credit\tAgreement,\tdated\tas\tof\tJanuary\t20,\t2023,\namong\tTesla,\tInc.,\tthe\tLenders\tand\tIssuing\tBanks\nfrom\ttime\tto\ttime\tparty\tthereto,\tCitibank,\tN.A.,\tas\nAdministrative\tAgent\tand\tDeutsche\tBank\nSecurities,\tInc.,\tas\tSyndication\tAgent\n10-K001-3475610.59January\t31,\t2023\n21.1List\tof\tSubsidiaries\tof\tthe\tRegistrant————X\n23.1Consent\tof\tPricewaterhouseCoopers\tLLP,\nIndependent\tRegistered\tPublic\tAccounting\tFirm\n————X\n31.1Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tExecutive\tOfficer\n————X\n31.2Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tFinancial\tOfficer\n————X\n32.1*Section\t1350\tCertifications————X\n97Tesla,\tInc.\tClawback\tPolicy————X\n101.INSInline\tXBRL\tInstance\tDocument————X\n101.SCHInline\tXBRL\tTaxonomy\tExtension\tSchema\nDocument\n————X\n101.CALInline\tXBRL\tTaxonomy\tExtension\tCalculation\nLinkbase\tDocument.\n————X\n101.DEFInline\tXBRL\tTaxonomy\tExtension\tDefinition\nLinkbase\tDocument\n————X\n101.LABInline\tXBRL\tTaxonomy\tExtension\tLabel\tLinkbase\nDocument\n————X\n101.PREInline\tXBRL\tTaxonomy\tExtension\tPresentation\nLinkbase\tDocument\n————X\n104Cover\tPage\tInteractive\tData\tFile\t(formatted\tas\ninline\tXBRL\twith\tapplicable\ttaxonomy\textension\ninformation\tcontained\tin\tExhibits\t101)\n*Furnished\therewith\n**Indicates\ta\tmanagement\tcontract\tor\tcompensatory\tplan\tor\tarrangement\n†Confidential\ttreatment\thas\tbeen\trequested\tfor\tportions\tof\tthis\texhibit\n††Portions\tof\tthis\texhibit\thave\tbeen\tredacted\tin\tcompliance\twith\tRegulation\tS-K\tItem\t601(b)(10).\n110",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 54,
data:             "from": 1
data:           },
data:           "pageNumber": 112
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "6f1f4702-359b-4ab4-94ac-050cfa1ec63a",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "4.15Indenture,\tdated\tas\tof\tMay\t22,\t2013,\tby\tand\nbetween\tthe\tRegistrant\tand\tU.S.\tBank\tNational\nAssociation.\n8-K001-347564.1May\t22,\t2013\n97",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 52,
data:             "from": 48
data:           },
data:           "pageNumber": 99
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "730548da-ef6a-4315-9fab-96fc778dce60",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "4.15Indenture,\tdated\tas\tof\tMay\t22,\t2013,\tby\tand\nbetween\tthe\tRegistrant\tand\tU.S.\tBank\tNational\nAssociation.\n8-K001-347564.1May\t22,\t2013\n97",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 52,
data:             "from": 48
data:           },
data:           "pageNumber": 99
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "ae09af6a-a557-4d09-8e9a-c50061df8ecf",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     }
data:   ]
data: }

event: values
data: {
data:   "query": "What is this document about?",
data:   "route": "retrieve",
data:   "messages": [
data:     {
data:       "content": "What is this document about?",
data:       "additional_kwargs": {},
data:       "response_metadata": {},
data:       "id": "a7cbf64c-3fcb-496f-982f-f6b00615f2ec",
data:       "type": "human"
data:     },
data:     {
data:       "content": "The document is a collection of exhibits from Tesla's 10-K filing for 2023. It includes various agreements, certifications, and lists, such as a grant contract for land use rights, a credit agreement, and certifications of executive officers. Additionally, it contains XBRL taxonomy extension documents and other regulatory compliance materials.",
data:       "additional_kwargs": {},
data:       "response_metadata": {
data:         "usage": {
data:           "prompt_tokens": 1603,
data:           "completion_tokens": 65,
data:           "total_tokens": 1668,
data:           "prompt_tokens_details": {
data:             "cached_tokens": 0,
data:             "audio_tokens": 0
data:           },
data:           "completion_tokens_details": {
data:             "reasoning_tokens": 0,
data:             "audio_tokens": 0,
data:             "accepted_prediction_tokens": 0,
data:             "rejected_prediction_tokens": 0
data:           }
data:         }
data:       },
data:       "tool_call_chunks": [],
data:       "id": "chatcmpl-CJFParD2fbaJZgJ7MQeAhUw8xS3Eq",
data:       "usage_metadata": {
data:         "input_tokens": 1603,
data:         "output_tokens": 65,
data:         "total_tokens": 1668,
data:         "input_token_details": {
data:           "audio": 0,
data:           "cache_read": 0
data:         },
data:         "output_token_details": {
data:           "audio": 0,
data:           "reasoning": 0
data:         }
data:       },
data:       "tool_calls": [],
data:       "invalid_tool_calls": [],
data:       "type": "ai"
data:     }
data:   ],
data:   "documents": [
data:     {
data:       "pageContent": "Exhibit\nNumber\nIncorporated\tby\tReference\nFiled\nHerewithExhibit\tDescriptionFormFile\tNo.ExhibitFiling\tDate\n10.58††Grant\tContract\tfor\tState-Owned\tConstruction\tLand\nUse\tRight,\tdated\tas\tof\tOctober\t17,\t2018,\tby\tand\nbetween\tShanghai\tPlanning\tand\tLand\tResource\nAdministration\tBureau,\tas\tgrantor,\tand\tTesla\n(Shanghai)\tCo.,\tLtd.,\tas\tgrantee\t(English\ntranslation).\n10-Q001-3475610.2July\t29,\t2019\n10.59Credit\tAgreement,\tdated\tas\tof\tJanuary\t20,\t2023,\namong\tTesla,\tInc.,\tthe\tLenders\tand\tIssuing\tBanks\nfrom\ttime\tto\ttime\tparty\tthereto,\tCitibank,\tN.A.,\tas\nAdministrative\tAgent\tand\tDeutsche\tBank\nSecurities,\tInc.,\tas\tSyndication\tAgent\n10-K001-3475610.59January\t31,\t2023\n21.1List\tof\tSubsidiaries\tof\tthe\tRegistrant————X\n23.1Consent\tof\tPricewaterhouseCoopers\tLLP,\nIndependent\tRegistered\tPublic\tAccounting\tFirm\n————X\n31.1Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tExecutive\tOfficer\n————X\n31.2Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tFinancial\tOfficer\n————X\n32.1*Section\t1350\tCertifications————X\n97Tesla,\tInc.\tClawback\tPolicy————X\n101.INSInline\tXBRL\tInstance\tDocument————X\n101.SCHInline\tXBRL\tTaxonomy\tExtension\tSchema\nDocument\n————X\n101.CALInline\tXBRL\tTaxonomy\tExtension\tCalculation\nLinkbase\tDocument.\n————X\n101.DEFInline\tXBRL\tTaxonomy\tExtension\tDefinition\nLinkbase\tDocument\n————X\n101.LABInline\tXBRL\tTaxonomy\tExtension\tLabel\tLinkbase\nDocument\n————X\n101.PREInline\tXBRL\tTaxonomy\tExtension\tPresentation\nLinkbase\tDocument\n————X\n104Cover\tPage\tInteractive\tData\tFile\t(formatted\tas\ninline\tXBRL\twith\tapplicable\ttaxonomy\textension\ninformation\tcontained\tin\tExhibits\t101)\n*Furnished\therewith\n**Indicates\ta\tmanagement\tcontract\tor\tcompensatory\tplan\tor\tarrangement\n†Confidential\ttreatment\thas\tbeen\trequested\tfor\tportions\tof\tthis\texhibit\n††Portions\tof\tthis\texhibit\thave\tbeen\tredacted\tin\tcompliance\twith\tRegulation\tS-K\tItem\t601(b)(10).\n110",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 54,
data:             "from": 1
data:           },
data:           "pageNumber": 112
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "8717aa08-52f2-4e9f-a857-4fd69d28028c",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "Exhibit\nNumber\nIncorporated\tby\tReference\nFiled\nHerewithExhibit\tDescriptionFormFile\tNo.ExhibitFiling\tDate\n10.58††Grant\tContract\tfor\tState-Owned\tConstruction\tLand\nUse\tRight,\tdated\tas\tof\tOctober\t17,\t2018,\tby\tand\nbetween\tShanghai\tPlanning\tand\tLand\tResource\nAdministration\tBureau,\tas\tgrantor,\tand\tTesla\n(Shanghai)\tCo.,\tLtd.,\tas\tgrantee\t(English\ntranslation).\n10-Q001-3475610.2July\t29,\t2019\n10.59Credit\tAgreement,\tdated\tas\tof\tJanuary\t20,\t2023,\namong\tTesla,\tInc.,\tthe\tLenders\tand\tIssuing\tBanks\nfrom\ttime\tto\ttime\tparty\tthereto,\tCitibank,\tN.A.,\tas\nAdministrative\tAgent\tand\tDeutsche\tBank\nSecurities,\tInc.,\tas\tSyndication\tAgent\n10-K001-3475610.59January\t31,\t2023\n21.1List\tof\tSubsidiaries\tof\tthe\tRegistrant————X\n23.1Consent\tof\tPricewaterhouseCoopers\tLLP,\nIndependent\tRegistered\tPublic\tAccounting\tFirm\n————X\n31.1Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tExecutive\tOfficer\n————X\n31.2Rule\t13a-14(a)\t/\t15(d)-14(a)\tCertification\tof\nPrincipal\tFinancial\tOfficer\n————X\n32.1*Section\t1350\tCertifications————X\n97Tesla,\tInc.\tClawback\tPolicy————X\n101.INSInline\tXBRL\tInstance\tDocument————X\n101.SCHInline\tXBRL\tTaxonomy\tExtension\tSchema\nDocument\n————X\n101.CALInline\tXBRL\tTaxonomy\tExtension\tCalculation\nLinkbase\tDocument.\n————X\n101.DEFInline\tXBRL\tTaxonomy\tExtension\tDefinition\nLinkbase\tDocument\n————X\n101.LABInline\tXBRL\tTaxonomy\tExtension\tLabel\tLinkbase\nDocument\n————X\n101.PREInline\tXBRL\tTaxonomy\tExtension\tPresentation\nLinkbase\tDocument\n————X\n104Cover\tPage\tInteractive\tData\tFile\t(formatted\tas\ninline\tXBRL\twith\tapplicable\ttaxonomy\textension\ninformation\tcontained\tin\tExhibits\t101)\n*Furnished\therewith\n**Indicates\ta\tmanagement\tcontract\tor\tcompensatory\tplan\tor\tarrangement\n†Confidential\ttreatment\thas\tbeen\trequested\tfor\tportions\tof\tthis\texhibit\n††Portions\tof\tthis\texhibit\thave\tbeen\tredacted\tin\tcompliance\twith\tRegulation\tS-K\tItem\t601(b)(10).\n110",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 54,
data:             "from": 1
data:           },
data:           "pageNumber": 112
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "6f1f4702-359b-4ab4-94ac-050cfa1ec63a",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "4.15Indenture,\tdated\tas\tof\tMay\t22,\t2013,\tby\tand\nbetween\tthe\tRegistrant\tand\tU.S.\tBank\tNational\nAssociation.\n8-K001-347564.1May\t22,\t2013\n97",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 52,
data:             "from": 48
data:           },
data:           "pageNumber": 99
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "730548da-ef6a-4315-9fab-96fc778dce60",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     },
data:     {
data:       "pageContent": "4.15Indenture,\tdated\tas\tof\tMay\t22,\t2013,\tby\tand\nbetween\tthe\tRegistrant\tand\tU.S.\tBank\tNational\nAssociation.\n8-K001-347564.1May\t22,\t2013\n97",
data:       "metadata": {
data:         "loc": {
data:           "lines": {
data:             "to": 52,
data:             "from": 48
data:           },
data:           "pageNumber": 99
data:         },
data:         "pdf": {
data:           "info": {
data:             "Title": "",
data:             "Creator": "wkhtmltopdf 0.12.6",
data:             "Producer": "Qt 5.15.2",
data:             "CreationDate": "D:20240129111114Z",
data:             "IsXFAPresent": false,
data:             "PDFFormatVersion": "1.4",
data:             "IsAcroFormPresent": false
data:           },
data:           "version": "1.10.100",
data:           "metadata": null,
data:           "totalPages": 130
data:         },
data:         "uuid": "ae09af6a-a557-4d09-8e9a-c50061df8ecf",
data:         "source": "./test_docs/test-tsla-10k-2023.pdf"
data:       }
data:     }
data:   ]
data: }

```

#### 랭스미스 UI를 통한 배포
https://smith.langchain.com/o/8af6585a-8ec8-4c69-8b66-b641e6802ab2/host/deployments 활용..(유료)

#### 랭그래프 스튜디오 실행
https://smith.langchain.com/studio/thread?organizationId=8af6585a-8ec8-4c69-8b66-b641e6802ab2&render=interact&baseUrl=http%3A%2F%2Flocalhost%3A2024&threadId=2eb1782e-6786-4f9a-8542-e1fcdfcc3bbe&mode=graph&assistantId=571ade52-f5cd-582a-89d9-d79dc861a8ba

### 보안
- 권한 제한
- 오용 가능성 예측
- 심층 방어
- 파일 접근
- API 접근
- 데이터베이스 접근
- 계정 생성 인증
- 요청 제한
- 프롬프트 주입 예약

In [8]:
// demo of the compiled graph running using the sdk
import { Client } from 'npm:langchain/langgraph-sdk';

import dotenv from 'npm:dotenv';

// Load environment variables from .env file
dotenv.config();

// Environment variables needed:
// LANGGRAPH_API_URL: The URL where your LangGraph server is running
//   - For local development: http://localhost:2024 (or your local server port)
//   - For LangSmith cloud: https://api.smith.langchain.com
//

const assistant_id = 'retrieval_graph';
async function runDemo() {
  // Initialize the LangGraph client
  const client = new Client({
    apiUrl: process.env.LANGGRAPH_API_URL || 'http://localhost:2024',
  });

  // Create a new thread for this conversation
  console.log('Creating new thread...');
  const thread = await client.threads.create({
    metadata: {
      demo: 'retrieval-graph',
    },
  });
  console.log('Thread created with ID:', thread.thread_id);

  // Example question
  const question = 'What is this document about?';

  console.log('\n=== Streaming Example ===');
  console.log('Question:', question);

  // Run the graph with streaming
  try {
    console.log('\nStarting stream...');
    const stream = await client.runs.stream(thread.thread_id, assistant_id, {
      input: { query: question },
      streamMode: ['values', 'messages', 'updates'], // Include all stream types
    });

    // Process the stream chunks
    console.log('\nWaiting for stream chunks...');
    for await (const chunk of stream) {
      console.log('\nReceived chunk:');
      console.log('Event type:', chunk.event);
      if (chunk.event === 'values') {
        console.log('Values data:', JSON.stringify(chunk.data, null, 2));
      } else if (chunk.event === 'messages/partial') {
        console.log('Messages data:', JSON.stringify(chunk, null, 2));
      } else if (chunk.event === 'updates') {
        console.log('Update data:', JSON.stringify(chunk.data, null, 2));
      }
    }
    console.log('\nStream completed.');
  } catch (error) {
    console.error('Error in streaming run:', error);
    // Log more details about the error
    if (error instanceof Error) {
      console.error('Error message:', error.message);
      console.error('Error stack:', error.stack);
    }
  }
}

// Run the demo
runDemo().catch((error) => {
  console.error('Fatal error:', error);
  process.exit(1);
});

TypeError: [ERR_MODULE_NOT_FOUND] Cannot find module 'file:///Users/user/Library/Caches/deno/npm/registry.npmjs.org/langchain/0.3.34_1/langgraph-sdk' imported from 'file:///Users/user/hs-dev/python/$deno$repl.mts'